<a href="https://colab.research.google.com/github/mejia080902-bit/pymc-examples/blob/main/markov_un_servidor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy
import matplotlib.pyplot as plt
from random import expovariate
from statistics import mean
from math import inf

# --- Parámetros del sistema M/M/1 ---
lamda = 4.0  # Tasa de llegadas (clientes/unidad de tiempo)
mu = 6.0     # Tasa de servicio (clientes/unidad de tiempo)
rho = lamda / mu # Factor de utilización

Num_Pkst = 100000 # Número de paquetes a simular
# T = 100000

# --- Variables de la simulación ---
count = 0  # Contador de salidas (paquetes atendidos)
t = 0      # Tiempo actual de la simulación
N = 0      # Número de clientes en el sistema

tll = expovariate(lamda)  # Tiempo de la próxima llegada
ts = inf                  # Tiempo de la próxima salida

t_evento_anterior = 0.0  # Tiempo del evento anterior

# --- Variables de salida existentes ---
tll_Data = []
ts_Data = []
r_Data = []
area = []

# --- Variables para el cálculo de Pn ---
# Diccionario para almacenar el tiempo total que el sistema tiene 'n' clientes.
# La clave es 'n' (el número de clientes) y el valor es el tiempo acumulado.
tiempo_en_estado = {0: 0.0}

# --- Bucle de simulación ---
while count < Num_Pkst:
#while t < T:
    # Registra el estado (N) y el tiempo que duró ese estado
    delta_t = t - t_evento_anterior
    # Suma el tiempo 'delta_t' al estado 'N' anterior
    if N not in tiempo_en_estado:
        tiempo_en_estado[N] = 0.0
    tiempo_en_estado[N] += delta_t

    # Define el próximo evento y actualiza el tiempo t
    if tll < ts:                     # Evento de llegada
        t = tll
        tll_Data.append(t)

        # Calculo de área (para L)
        area.append((t - t_evento_anterior) * N)
        t_evento_anterior = t

        N = N + 1.0
        tll = t + expovariate(lamda)
        if N == 1:
            ts = t + expovariate(mu)
    else:                            # Evento de salida
        t = ts
        ts_Data.append(t)

        # Calculo de área (para L)
        area.append((t - t_evento_anterior) * N)
        t_evento_anterior = t

        N = N - 1.0
        count = count + 1
        if N > 0:
            ts = t + expovariate(mu)
        else:
            ts = inf

# --- Paso final para el último estado ---
# El último estado (N) duró desde t_evento_anterior hasta el tiempo final de la simulación (t).
delta_t = t - t_evento_anterior
if N not in tiempo_en_estado:
    tiempo_en_estado[N] = 0.0
tiempo_en_estado[N] += delta_t


# --- Estimación del Retraso y L ---
for i in range(Num_Pkst):
    d = ts_Data[i] - tll_Data[i]
    r_Data.append(d)

print("=== Resultados de la Simulación M/M/1 ===")
print(f"Parámetros: lambda={lamda}, mu={mu}, rho={rho:.4f}")
print("--- Métricas de Desempeño (M/M/1) ---")
print("Retraso promedio = W (Simulación) = ", round(mean(r_Data), 4))
print("E[N(t)] = L (Simulación) = ", round(sum(area)/t, 4))

# --- Cálculo y Muestra de Probabilidades Pn ---
tiempo_total_simulacion = t
probabilidades_simulacion = {}

print("\n--- Probabilidades de Estado Pn (Simulación) ---")

# Obtenemos la lista de estados N ordenados
estados_ordenados = sorted(tiempo_en_estado.keys())

for n in estados_ordenados:
    tiempo = tiempo_en_estado[n]
    # Pn = (Tiempo en el estado n) / (Tiempo total de simulación)
    prob_n = tiempo / tiempo_total_simulacion
    probabilidades_simulacion[n] = prob_n
    prob_n_teorica = (1.0 - rho) * (rho**n)

    print(f"P({int(n):<2} clientes): Teórico={prob_n_teorica:.6f}")


=== Resultados de la Simulación M/M/1 ===
Parámetros: lambda=4.0, mu=6.0, rho=0.6667
--- Métricas de Desempeño (M/M/1) ---
Retraso promedio = W (Simulación) =  0.4939
E[N(t)] = L (Simulación) =  1.9756

--- Probabilidades de Estado Pn (Simulación) ---
P(0  clientes): Teórico=0.333333
P(1  clientes): Teórico=0.222222
P(2  clientes): Teórico=0.148148
P(3  clientes): Teórico=0.098765
P(4  clientes): Teórico=0.065844
P(5  clientes): Teórico=0.043896
P(6  clientes): Teórico=0.029264
P(7  clientes): Teórico=0.019509
P(8  clientes): Teórico=0.013006
P(9  clientes): Teórico=0.008671
P(10 clientes): Teórico=0.005781
P(11 clientes): Teórico=0.003854
P(12 clientes): Teórico=0.002569
P(13 clientes): Teórico=0.001713
P(14 clientes): Teórico=0.001142
P(15 clientes): Teórico=0.000761
P(16 clientes): Teórico=0.000507
P(17 clientes): Teórico=0.000338
P(18 clientes): Teórico=0.000226
P(19 clientes): Teórico=0.000150
P(20 clientes): Teórico=0.000100
P(21 clientes): Teórico=0.000067
P(22 clientes): Teóric